# Class Associations in Python: `Item` and `Owner`

This short workshop notebook develops class relationships from a basic association to one-to-one, one-to-many, aggregation, and composition. It is designed to fit within **10 presentation pages/sections** when rendered.

## Learning outcomes
By the end, you should be able to model and implement associations between two Python classes, explain multiplicity, and distinguish aggregation from composition.

## Table of contents
1. [Scenario and class diagrams](#scenario-and-class-diagrams)
2. [Step 1: Basic association](#step-1-basic-association)
3. [Step 2: One-to-one association](#step-2-one-to-one-association)
4. [Step 3: One-to-many association](#step-3-one-to-many-association)
5. [Step 4: Aggregation](#step-4-aggregation)
6. [Step 5: Composition](#step-5-composition)
7. [Activity](#activity)


## Scenario and class diagrams

An **Owner** is a person or organisation that can be linked to an **Item**. We will use the same two classes throughout, enriching the relationship each time. Each class has two attributes and three methods.

```mermaid
classDiagram
    class Owner {
      +str name
      +str owner_id
      +introduce() str
      +change_name(new_name) None
      +describe() str
    }
    class Item {
      +str title
      +str item_id
      +rename(new_title) None
      +label() str
      +describe() str
    }
```

**Multiplicity notation:** `1` means exactly one; `0..*` means zero or more. A plain line is an association, an open diamond is aggregation, and a filled diamond is composition.


## Step 1: Basic association

A basic association means two objects know about each other through a reference. At this stage, an `Item` stores one optional owner; the `Owner` does not yet maintain a collection of items.

```mermaid
classDiagram
    Owner "0..1" <-- "0..1" Item : associated with
```


In [ ]:
class Owner:
    def __init__(self, name, owner_id):
        self.name = name
        self.owner_id = owner_id

    def introduce(self):
        return f"I am {self.name}."

    def change_name(self, new_name):
        self.name = new_name

    def describe(self):
        return f"Owner({self.owner_id}, {self.name})"


class Item:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.owner = None  # association reference

    def rename(self, new_title):
        self.title = new_title

    def label(self):
        return f"{self.item_id}: {self.title}"

    def describe(self):
        owner_name = self.owner.name if self.owner else "unassigned"
        return f"Item({self.label()}, owner={owner_name})"

owner_a = Owner("Amina", "O-01")
item_a = Item("Laptop", "I-01")
item_a.owner = owner_a
print(item_a.describe())


## Step 2: One-to-one association

In a **one-to-one** relationship, one owner is linked to one item, and the item is linked back to that same owner. A helper method keeps both ends consistent.

```mermaid
classDiagram
    Owner "1" <--> "1" Item : owns
```


In [ ]:
class OneToOneOwner(Owner):
    def __init__(self, name, owner_id):
        super().__init__(name, owner_id)
        self.item = None

    def assign_item(self, item):
        self.item = item
        item.owner = self

one_owner = OneToOneOwner("Ben", "O-02")
one_item = Item("Security pass", "I-02")
one_owner.assign_item(one_item)
print(one_owner.describe())
print(one_item.describe())


## Step 3: One-to-many association

In a **one-to-many** relationship, one `Owner` manages many `Item` objects, while each item has one owner. The owner uses `add_item` and `remove_item` to manage the association.

```mermaid
classDiagram
    Owner "1" <--> "0..*" Item : manages
```


In [ ]:
class PortfolioOwner(Owner):
    def __init__(self, name, owner_id):
        super().__init__(name, owner_id)
        self.items = []

    def add_item(self, item):
        if item not in self.items:
            self.items.append(item)
            item.owner = self

    def remove_item(self, item):
        if item in self.items:
            self.items.remove(item)
            item.owner = None

    def list_items(self):
        return [item.label() for item in self.items]

portfolio_owner = PortfolioOwner("Chen", "O-03")
portfolio_owner.add_item(Item("Tablet", "I-03"))
portfolio_owner.add_item(Item("Headset", "I-04"))
print(portfolio_owner.list_items())


## Step 4: Aggregation

**Aggregation** is a weak whole-part relationship: the owner groups existing items, but items can still exist independently and can be moved to another owner. The open diamond is placed at the whole (`Owner`) end.

```mermaid
classDiagram
    Owner "1" o-- "0..*" Item : aggregates
```


In [ ]:
shared_item = Item("Projector", "I-05")  # created independently
team_a = PortfolioOwner("Design Team", "O-04")
team_b = PortfolioOwner("Teaching Team", "O-05")

team_a.add_item(shared_item)
team_a.remove_item(shared_item)  # item survives after removal
team_b.add_item(shared_item)

print(shared_item.describe())
print(team_b.list_items())


## Step 5: Composition

**Composition** is a strong whole-part relationship: the owner creates and controls its items. When the owner closes its collection, the owned items are cleared from that owner’s lifecycle; they are not transferred elsewhere through this model.

```mermaid
classDiagram
    Owner "1" *-- "0..*" Item : composes
```


In [ ]:
class CollectionOwner(Owner):
    def __init__(self, name, owner_id):
        super().__init__(name, owner_id)
        self._items = []

    def create_item(self, title, item_id):
        item = Item(title, item_id)
        item.owner = self
        self._items.append(item)
        return item

    def list_items(self):
        return [item.label() for item in self._items]

    def close_collection(self):
        for item in self._items:
            item.owner = None
        self._items.clear()

collection = CollectionOwner("Museum Store", "O-06")
catalogue_item = collection.create_item("Exhibition catalogue", "I-06")
print(collection.list_items())
collection.close_collection()
print(collection.list_items(), catalogue_item.owner)


## Activity

Work with a **Book** and **Author** model using only the two classes below. Start from the `Owner`/`Item` implementations in this notebook, then adapt them to `Author` and `Book` as described.

### Classes to use

- `Author`
  - Attributes: `name` (str), `author_id` (str)
  - Methods: `introduce()` → str, `change_name(new_name)` → None, `describe()` → str

- `Book`
  - Attributes: `title` (str), `book_id` (str)
  - Methods: `rename(new_title)` → None, `label()` → str, `describe()` → str

You may rename `Owner` → `Author` and `Item` → `Book` in your code, or subclass them.

### Tasks

1. Create an `Author` called `"A. N. Author"` and three books: `"First Draft"`, `"Second Draft"`, and `"Final Manuscript"`.
2. Implement a **one-to-many** relationship so `"A. N. Author"` can list all three books, and each book can identify its author.
3. Remove `"First Draft"` from the author’s collection and show that it still exists as an independent object: this demonstrates **aggregation**.
4. Create a second author that uses a **composition-style** `create_book` method to make a new `"Workbook"` book. Close that author’s collection and verify that its book list is empty.
